# Stress Prediction — Combined Final 0806 재튜닝

이전 제출물 `submit_combined_final_0806_0.24.csv` (gender 제거 + winsorize 4개 + 근접중복 오버라이드 + PAIR_WEIGHT=0.24, CV 0.147300)에서
남아있던 튜닝 여지를 정리합니다.

- `PAIR_WEIGHT` 탐색 범위(0.24~0.45)의 **경계값(0.24)이 최적으로 나왔는데도 그보다 낮은 값은 확인되지 않았음**
- `TREE_QUANTILE`(0.54), `PAIR_QUANTILE`(0.48), `OVERRIDE_THRESHOLD`(0.2)는 이전 노트북에서 한 번도 튜닝되지 않았음

이 노트북은 폴드별 원시 예측(ExtraTrees 개별 트리 예측, pair 이웃 타겟 행렬)을 **한 번만 계산해 캐시**해두고,
재학습 없이 `TREE_QUANTILE` · `PAIR_QUANTILE` · `PAIR_WEIGHT` · `OVERRIDE_THRESHOLD`를 순차적으로(coordinate search) 재탐색합니다.
매 스텝은 Train-only 5-Fold CV로만 평가하며, Test 데이터는 최종 제출 파일을 만들 때만 사용합니다.

**결과**: CV MAE `0.147300` → **`0.147090`** (개선폭 `-0.000210`)
최종 설정: `tree_q=0.54, pair_q=0.42, pair_weight=0.24, override_threshold=0.15`

주의: 팀의 로컬-퍼블릭 격차는 통상 약 0.02 수준이고 변동폭도 있어서, 이 CV 개선이 Public 점수 개선으로
그대로 이어진다는 보장은 없습니다. 기존 구조를 최대한 짜낸 미세 튜닝이며, 큰 폭의 돌파구는 아닙니다.


In [ ]:
from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200

# 최종 채택 값 (이 노트북에서 재탐색한 결과). 괄호 안은 이전 노트북(0806_1)의 값.
TREE_QUANTILE = 0.54        # (이전: 0.54, 변화 없음)
PAIR_QUANTILE = 0.42        # (이전: 0.48)
PAIR_WEIGHT = 0.24          # (이전: 0.24, 변화 없음 — 여전히 최적)
OVERRIDE_THRESHOLD = 0.15   # (이전: 0.20)
WINSORIZE_COLUMNS = ["mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def compute_iqr_bounds(train_frame, columns):
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_raw_predictions(train_features, val_features, target) -> np.ndarray:
    """한 번만 fit하고, 개별 트리 예측 행렬(n_val, n_estimators)을 그대로 반환합니다."""
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    return np.column_stack([t.predict(val_matrix) for t in model.estimators_])


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_targets(train_features, val_features, target) -> np.ndarray:
    """모든 컬럼쌍(28개)에 대한 최근접 이웃 타겟 행렬(n_val, n_pairs)을 반환합니다."""
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.column_stack(neighbor_targets)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def combined_prediction(train_raw_fold, val_raw_fold, target, numeric_cols,
                         tree_q=TREE_QUANTILE, pair_q=PAIR_QUANTILE,
                         pair_weight=PAIR_WEIGHT, override_threshold=OVERRIDE_THRESHOLD):
    """gender 제거 + winsorize + 근접중복 오버라이드 전부 적용 (최종 설정 기본값)."""
    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)

    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_raw = tree_raw_predictions(train_feat, val_feat, target)
    pair_raw = pair_neighbor_targets(train_feat, val_feat, target)
    tree_pred = np.quantile(tree_raw, tree_q, axis=1)
    pair_pred = np.quantile(pair_raw, pair_q, axis=1)
    base_pred = np.clip(np.round((1 - pair_weight) * tree_pred + pair_weight * pair_pred, 2), 0.0, 1.0)

    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, target, numeric_cols)
    override_mask = nearest_dist < override_threshold
    result = base_pred.copy()
    result[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
    return result


print("설정 로드 완료:", dict(tree_q=TREE_QUANTILE, pair_q=PAIR_QUANTILE,
                          pair_weight=PAIR_WEIGHT, override_threshold=OVERRIDE_THRESHOLD))


설정 로드 완료: {'tree_q': 0.54, 'pair_q': 0.42, 'pair_weight': 0.24, 'override_threshold': 0.15}


## 하이퍼파라미터 재탐색 (폴드별 원시 예측 캐시, 재학습 없음)

`tree_raw_predictions`(트리별 예측)와 `pair_neighbor_targets`(쌍별 이웃 타겟)는 fold당 한 번만 계산해
`fold_cache`에 저장합니다. 이후 `TREE_QUANTILE`/`PAIR_QUANTILE`/`PAIR_WEIGHT`/`OVERRIDE_THRESHOLD` 조합은
캐시된 배열에 `np.quantile`과 가중합만 다시 적용해 빠르게 평가합니다(재학습 없음).

탐색 순서: ① PAIR_WEIGHT를 이전 경계값(0.24) 아래로 확장 → ② TREE_QUANTILE × PAIR_QUANTILE 그리드 →
③ 새 quantile로 PAIR_WEIGHT 재탐색 → ④ OVERRIDE_THRESHOLD 탐색. 각 단계는 이전 단계에서 찾은 최적값을 고정하고 진행합니다(coordinate search).

In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)

fold_cache = []
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)

    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)
    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_raw = tree_raw_predictions(train_feat, val_feat, y_train)
    pair_raw = pair_neighbor_targets(train_feat, val_feat, y_train)
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, y_train, numeric_cols)

    fold_cache.append((val_idx, tree_raw, pair_raw, nearest_dist, nearest_target, y_val))
    print(f"Fold {fold} 캐시 완료")


def evaluate(tree_q, pair_q, pair_weight, override_threshold):
    oof = np.zeros(len(y))
    for val_idx, tree_raw, pair_raw, nearest_dist, nearest_target, y_val in fold_cache:
        tree_pred = np.quantile(tree_raw, tree_q, axis=1)
        pair_pred = np.quantile(pair_raw, pair_q, axis=1)
        pred = np.clip(np.round((1 - pair_weight) * tree_pred + pair_weight * pair_pred, 2), 0.0, 1.0)
        override_mask = nearest_dist < override_threshold
        pred[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
        oof[val_idx] = pred
    return mean_absolute_error(y, oof)


baseline = evaluate(0.54, 0.48, 0.28, 0.2)
print(f"\n[sanity check] tree_q=0.54 pair_q=0.48 pair_w=0.28 thr=0.2 -> {baseline:.6f} (0806_1 노트북 기록값 0.147363)")

print("\n=== ① PAIR_WEIGHT 확장 탐색 (tree_q=0.54, pair_q=0.48, thr=0.2) ===")
stage1 = []
for w in [0.0, 0.05, 0.10, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24]:
    mae = evaluate(0.54, 0.48, w, 0.2)
    stage1.append((w, mae))
    print(f"PAIR_WEIGHT={w:.2f}: {mae:.6f}")
stage1.sort(key=lambda r: r[1])
best_w = stage1[0][0]
print(f"-> 최적: PAIR_WEIGHT={best_w} ({stage1[0][1]:.6f})")

print(f"\n=== ② TREE_QUANTILE x PAIR_QUANTILE 그리드 (pair_w={best_w}, thr=0.2) ===")
stage2 = []
for tq in [0.48, 0.50, 0.52, 0.54, 0.56, 0.58]:
    for pq in [0.42, 0.44, 0.46, 0.48, 0.50, 0.52]:
        mae = evaluate(tq, pq, best_w, 0.2)
        stage2.append((tq, pq, mae))
stage2.sort(key=lambda r: r[2])
for tq, pq, mae in stage2[:5]:
    print(f"tree_q={tq:.2f} pair_q={pq:.2f}: {mae:.6f}")
best_tq, best_pq, _ = stage2[0]
print(f"-> 최적: tree_q={best_tq} pair_q={best_pq}")

print(f"\n=== ③ PAIR_WEIGHT 재탐색 (tree_q={best_tq}, pair_q={best_pq}, thr=0.2) ===")
stage3 = []
for w in [0.0, 0.05, 0.10, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24, 0.28, 0.32]:
    mae = evaluate(best_tq, best_pq, w, 0.2)
    stage3.append((w, mae))
    print(f"PAIR_WEIGHT={w:.2f}: {mae:.6f}")
stage3.sort(key=lambda r: r[1])
best_w2 = stage3[0][0]
print(f"-> 최적: PAIR_WEIGHT={best_w2} ({stage3[0][1]:.6f})")

print(f"\n=== ④ OVERRIDE_THRESHOLD 탐색 (tree_q={best_tq}, pair_q={best_pq}, pair_w={best_w2}) ===")
stage4 = []
for thr in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    mae = evaluate(best_tq, best_pq, best_w2, thr)
    stage4.append((thr, mae))
    print(f"OVERRIDE_THRESHOLD={thr}: {mae:.6f}")
stage4.sort(key=lambda r: r[1])
best_thr, best_final_mae = stage4[0]
print(f"-> 최적: OVERRIDE_THRESHOLD={best_thr} ({best_final_mae:.6f})")

print("\n================ 최종 채택 설정 ================")
print(f"tree_q={best_tq}, pair_q={best_pq}, pair_weight={best_w2}, override_threshold={best_thr}")
print(f"CV MAE: {best_final_mae:.6f}  (이전 0.24 설정 대비 개선폭: {0.147300 - best_final_mae:+.6f})")


Fold 1 캐시 완료
Fold 2 캐시 완료
Fold 3 캐시 완료
Fold 4 캐시 완료
Fold 5 캐시 완료

[sanity check] tree_q=0.54 pair_q=0.48 pair_w=0.28 thr=0.2 -> 0.147363 (0806_1 노트북 기록값 0.147363)

=== ① PAIR_WEIGHT 확장 탐색 (tree_q=0.54, pair_q=0.48, thr=0.2) ===
PAIR_WEIGHT=0.00: 0.147763
PAIR_WEIGHT=0.05: 0.147573
PAIR_WEIGHT=0.10: 0.147430
PAIR_WEIGHT=0.14: 0.147367
PAIR_WEIGHT=0.16: 0.147310
PAIR_WEIGHT=0.18: 0.147357
PAIR_WEIGHT=0.20: 0.147360
PAIR_WEIGHT=0.22: 0.147350
PAIR_WEIGHT=0.24: 0.147300
-> 최적: PAIR_WEIGHT=0.24 (0.147300)

=== ② TREE_QUANTILE x PAIR_QUANTILE 그리드 (pair_w=0.24, thr=0.2) ===
tree_q=0.54 pair_q=0.42: 0.147090
tree_q=0.52 pair_q=0.48: 0.147130
tree_q=0.56 pair_q=0.42: 0.147193
tree_q=0.52 pair_q=0.50: 0.147247
tree_q=0.52 pair_q=0.42: 0.147250
-> 최적: tree_q=0.54 pair_q=0.42

=== ③ PAIR_WEIGHT 재탐색 (tree_q=0.54, pair_q=0.42, thr=0.2) ===
PAIR_WEIGHT=0.00: 0.147763
PAIR_WEIGHT=0.05: 0.147537
PAIR_WEIGHT=0.10: 0.147297
PAIR_WEIGHT=0.14: 0.147223
PAIR_WEIGHT=0.16: 0.147270
PAIR_WEIGHT=0.18: 0.147170


## 5-Fold CV — 최종 설정 검증

위에서 찾은 최종 설정(`TREE_QUANTILE=0.54, PAIR_QUANTILE=0.42, PAIR_WEIGHT=0.24, OVERRIDE_THRESHOLD=0.15`)을
`combined_prediction` 기본값으로 다시 한 번 처음부터(재학습 포함) 5-Fold CV로 재검증합니다.

In [ ]:
oof_final = np.zeros(len(y))
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    oof_final[val_idx] = combined_prediction(train_raw_fold, val_raw_fold, y_train, numeric_cols)
    print(f"Fold {fold} done")

final_cv_mae = mean_absolute_error(y, oof_final)
print()
print(f"### 최종 설정 CV MAE: {final_cv_mae:.6f}")
print("(참고) 이전 0.24 설정 CV MAE: 0.147300")
print(f"(참고) 개선폭: {0.147300 - final_cv_mae:+.6f}")


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

### 최종 설정 CV MAE: 0.147090
(참고) 이전 0.24 설정 CV MAE: 0.147300
(참고) 개선폭: +0.000210


## 최종 제출 파일 생성

In [ ]:
test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)
test_ids = test_raw[ID_COLUMN]

final_prediction = combined_prediction(raw_input, test_input, y, numeric_cols)

output = submission.copy()
output[ID_COLUMN] = test_ids
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_combined_final_0806_retuned.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("사용된 설정:", dict(tree_q=TREE_QUANTILE, pair_q=PAIR_QUANTILE,
                        pair_weight=PAIR_WEIGHT, override_threshold=OVERRIDE_THRESHOLD))
print("Prediction range:", final_prediction.min(), final_prediction.max())
print(output.head(10))

# 이전 제출물과 비교
prior = pd.read_csv(OUTPUT_DIR / "submit_combined_final_0806_0.24.csv")
merged = prior.merge(output, on=ID_COLUMN, suffixes=("_prior", "_new"))
diff = (merged[f"{TARGET}_prior"] - merged[f"{TARGET}_new"]).abs()
print()
print(f"이전 제출 대비 변경된 행: {(diff > 1e-9).sum()} / {len(merged)}")
print(f"평균 절대 변화량: {diff.mean():.6f}, 최대 변화량: {diff.max():.6f}")

print()
print("=== VALIDATION ===")
checks = {
    "row_count_matches_test": len(output) == len(test_raw),
    "ids_match_test_exactly": list(output[ID_COLUMN]) == list(test_ids),
    "ids_unique": output[ID_COLUMN].is_unique,
    "no_missing_values": not output.isna().any().any(),
    "target_in_range_0_1": output[TARGET].between(0, 1).all(),
}
for k, v in checks.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_combined_final_0806_retuned.csv
사용된 설정: {'tree_q': 0.54, 'pair_q': 0.42, 'pair_weight': 0.24, 'override_threshold': 0.15}
Prediction range: 0.0 1.0
          ID  stress_score
0  TEST_0000          0.59
1  TEST_0001          0.97
2  TEST_0002          0.19
3  TEST_0003          0.52
4  TEST_0004          0.53
5  TEST_0005          0.49
6  TEST_0006          0.78
7  TEST_0007          0.47
8  TEST_0008          0.51
9  TEST_0009          0.88

이전 제출 대비 변경된 행: 1135 / 3000
평균 절대 변화량: 0.006667, 최대 변화량: 0.070000

=== VALIDATION ===
row_count_matches_test: PASS
ids_match_test_exactly: PASS
ids_unique: PASS
no_missing_values: PASS
target_in_range_0_1: PASS
